# AI PDF to CSV Automation
### Reproducible invoice extraction comparison and interactive app

This notebook compares rule-based Python extraction with Qwen 2.5 1.5B Instruct on five generated synthetic invoices, evaluates both methods against known source values, benchmarks runtime, exports CSV results, and launches a Gradio upload interface.

**Scope:** text-based PDFs; one invoice per page; scanned/image-only PDFs require OCR and are not supported here. The five research invoices are generated in this notebook so a fresh run does not depend on an untracked upload.

## 1. Install dependencies
Run once in a fresh Google Colab runtime.

In [ ]:
%pip -q install pymupdf pandas matplotlib transformers accelerate gradio

## 2. Imports and configuration

In [ ]:
import os, re, json, time, tempfile
import fitz
import pandas as pd
import matplotlib.pyplot as plt
import torch
import gradio as gr
from transformers import AutoTokenizer, AutoModelForCausalLM

pd.set_option("display.max_columns", None)
MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
FIELDS = ["Invoice_ID", "Invoice_Date", "Customer", "Product", "Quantity", "Unit_Price", "Total_Amount"]
print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## 3. Create a repeatable five-invoice test set
These synthetic records are the ground truth. Each PDF page contains one invoice with a 18% GST rate.

In [ ]:
GROUND_TRUTH = [
    {"Invoice_ID":"INV-2026-001","Invoice_Date":"10 November 2026","Customer":"ABC Learning Services","Product":"Office notebooks","Quantity":50,"Unit_Price":120.0,"Total_Amount":7080.0,"GST_Rate":0.18},
    {"Invoice_ID":"INV-2026-002","Invoice_Date":"12 November 2026","Customer":"Zenith Retail Group","Product":"Wireless barcode scanners","Quantity":4,"Unit_Price":5500.0,"Total_Amount":25960.0,"GST_Rate":0.18},
    {"Invoice_ID":"INV-2026-003","Invoice_Date":"14 November 2026","Customer":"PQR Food Traders","Product":"Recyclable shipping boxes","Quantity":120,"Unit_Price":45.0,"Total_Amount":6372.0,"GST_Rate":0.18},
    {"Invoice_ID":"INV-2026-004","Invoice_Date":"16 November 2026","Customer":"Cedar Consulting","Product":"Ergonomic office chairs","Quantity":3,"Unit_Price":8250.0,"Total_Amount":29205.0,"GST_Rate":0.18},
    {"Invoice_ID":"INV-2026-005","Invoice_Date":"18 November 2026","Customer":"Riverbend Training Centre","Product":"Whiteboard marker sets","Quantity":15,"Unit_Price":280.0,"Total_Amount":4956.0,"GST_Rate":0.18},
]

sample_pdf_path = "sample_invoices.pdf"
doc = fitz.open()
for row in GROUND_TRUTH:
    page = doc.new_page()
    lines = [
        "TAX INVOICE",
        f"Invoice Number: {row['Invoice_ID']}",
        f"Invoice Date: {row['Invoice_Date']}",
        f"Bill To: {row['Customer']}",
        f"Product: {row['Product']}",
        f"Quantity: {row['Quantity']}",
        f"Unit Price: INR {row['Unit_Price']:.2f}",
        f"Line Subtotal: INR {row['Quantity'] * row['Unit_Price']:.2f}",
        f"GST: {row['GST_Rate'] * 100:.0f}%",
        f"Grand Total Due: INR {row['Total_Amount']:.2f}",
    ]
    y = 65
    for line in lines:
        page.insert_text((55, y), line, fontsize=11)
        y += 28
doc.save(sample_pdf_path)
doc.close()
print("Created", sample_pdf_path, "with", len(GROUND_TRUTH), "invoice pages.")

## 4. Traditional extraction
Use explicit field labels rather than relying on a fixed table-row position. Missing values remain null and are reported.

In [ ]:
def parse_number(value):
    """Convert currency-like values to floats without losing decimal points."""
    if value is None:
        return None
    cleaned = re.sub(r"[^0-9.\\-]", "", str(value).replace(",", ""))
    try:
        return float(cleaned) if cleaned not in ("", ".", "-", "-.") else None
    except ValueError:
        return None


def next_value_after_label(lines, label_pattern):
    """
    Extract a value written on the same line as a label or on the next line.
    The non-capturing group is important when label_pattern contains alternatives.
    """
    pattern = re.compile(r"^(?:" + label_pattern + r")\\s*(?::|-)??\\s*(.*)$", re.IGNORECASE)
    for index, line in enumerate(lines):
        match = pattern.search(line.strip())
        if match:
            value = match.group(1).strip()
            if value:
                return value
            if index + 1 < len(lines):
                return lines[index + 1].strip()
    return None


def extract_traditional_invoice(page):
    """
    Rule-based extraction for text PDFs.
    Handles both inline labels (Grand Total Due: INR 100) and
    labels whose values appear on the following line (Invoice Number / Bill To).
    It also reads the item row after the Product / Quantity / Unit Price headers.
    """
    text = page.get_text("text")
    lines = [line.strip() for line in text.splitlines() if line.strip()]

    record = {
        "Invoice_ID": next_value_after_label(
            lines, r"Invoice\\s*(?:Number|No\\.?|ID|#)"
        ),
        "Invoice_Date": next_value_after_label(lines, r"Invoice\\s*Date"),
        "Customer": next_value_after_label(lines, r"Bill\\s*To|Billed\\s*To|Customer"),
        "Product": None,
        "Quantity": None,
        "Unit_Price": None,
        "Total_Amount": parse_number(next_value_after_label(
            lines, r"Grand\\s*Total\\s*Due|Total\\s*Amount\\s*Due|Amount\\s*Payable|Total\\s*Payable|Invoice\\s*Total"
        )),
    }

    # Locate the item table and read the first item row after its header labels.
    product_header = next(
        (i for i, line in enumerate(lines)
         if re.fullmatch(r"Product(?:\\s+Description)?", line, flags=re.IGNORECASE)),
        None
    )
    if product_header is not None:
        subtotal_header = next(
            (i for i in range(product_header + 1, min(product_header + 8, len(lines)))
             if re.search(r"Line\\s+Subtotal", lines[i], flags=re.IGNORECASE)),
            None
        )
        if subtotal_header is not None and subtotal_header + 3 < len(lines):
            record["Product"] = lines[subtotal_header + 1]
            record["Quantity"] = parse_number(lines[subtotal_header + 2])
            record["Unit_Price"] = parse_number(lines[subtotal_header + 3])

    return record


research_doc = fitz.open(sample_pdf_path)
traditional_records = [extract_traditional_invoice(page) for page in research_doc]
traditional_df = pd.DataFrame(traditional_records, columns=FIELDS)
display(traditional_df)
print("Traditional missing values:", int(traditional_df.isna().sum().sum()))

## 5. Load Qwen and extract with AI
The model is loaded once and reused. A Colab GPU is recommended; CPU inference can be slow.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto"
)
model.eval()
print("Loaded", MODEL_NAME, "on", model.device)

def normalize_ai_record(data):
    result = {}
    for field in FIELDS:
        value = data.get(field)
        if isinstance(value, (dict, list)):
            value = None
        if field in ["Quantity", "Unit_Price", "Total_Amount"]:
            result[field] = parse_number(value)
        else:
            result[field] = str(value).strip() if value is not None else None
    return result

def extract_invoice_with_ai(page):
    invoice_text = page.get_text("text")
    if not invoice_text.strip():
        raise ValueError("No selectable PDF text found; OCR is required for scanned PDFs.")
    prompt = f'''Extract the invoice fields as one flat JSON object with exactly these keys:
Invoice_ID, Invoice_Date, Customer, Product, Quantity, Unit_Price, Total_Amount.
Use null for missing values. Numeric fields must be numbers. Total_Amount is the final payable total including tax.
Return JSON only. Invoice text:\n{invoice_text}'''
    messages = [{"role":"user","content":prompt}]
    formatted = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(formatted, return_tensors="pt").to(model.device)
    with torch.inference_mode():
        output = model.generate(**inputs, max_new_tokens=250, do_sample=False)
    response = tokenizer.decode(output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    match = re.search(r"\{.*\}", response, flags=re.S)
    if not match:
        raise ValueError("Model did not return a JSON object: " + response[:300])
    raw = json.loads(match.group(0))
    if not isinstance(raw, dict):
        raise ValueError("Model response must be a JSON object.")
    record = normalize_ai_record(raw)
    # Prefer a source-text-labelled final amount over a model guess.
    labelled_total = labelled_value(invoice_text, r"Grand\s*Total\s*Due|Total\s*Amount\s*Due|Amount\s*Payable")
    if labelled_total is not None:
        record["Total_Amount"] = parse_number(labelled_total)
    return record

ai_records = [extract_invoice_with_ai(page) for page in research_doc]
ai_df = pd.DataFrame(ai_records, columns=FIELDS)
display(ai_df)
print("AI missing values:", int(ai_df.isna().sum().sum()))

## 6. Compare both methods against ground truth
A match is counted only when both values are present and equal (numeric fields use a small tolerance). This prevents two missing values from being counted as a successful match.

In [ ]:
truth_df = pd.DataFrame(GROUND_TRUTH)[FIELDS]
def evaluate_method(actual_df, truth_df, method):
    merged = truth_df.merge(actual_df, on="Invoice_ID", how="left", suffixes=("_truth", "_actual"))
    rows = []
    for _, row in merged.iterrows():
        for field in [f for f in FIELDS if f != "Invoice_ID"]:
            expected, actual = row[f"{field}_truth"], row[f"{field}_actual"]
            if pd.isna(expected) or pd.isna(actual):
                matched = False
            elif field in ["Quantity", "Unit_Price", "Total_Amount"]:
                matched = abs(float(expected) - float(actual)) < 0.01
            else:
                matched = str(expected).strip().casefold() == str(actual).strip().casefold()
            rows.append({"Method":method, "Invoice_ID":row["Invoice_ID"], "Field":field, "Match":matched})
    detail = pd.DataFrame(rows)
    return detail

traditional_eval = evaluate_method(traditional_df, truth_df, "Traditional Python")
ai_eval = evaluate_method(ai_df, truth_df, "AI Extraction")
field_comparison = pd.concat([traditional_eval, ai_eval], ignore_index=True)
summary = field_comparison.groupby("Method")["Match"].agg(["sum","count"]).reset_index()
summary["Accuracy (%)"] = (summary["sum"] / summary["count"] * 100).round(2)
summary.rename(columns={"sum":"Successful Fields","count":"Total Fields"}, inplace=True)
display(summary)
display(field_comparison.pivot_table(index=["Invoice_ID","Field"], columns="Method", values="Match"))

## 7. Independent financial validation
For these generated test invoices, calculate the expected total from quantity × unit price × (1 + GST rate), then compare each method's extracted amount with that independently calculated value. This check applies to the stated 18% synthetic test data, not arbitrary real-world tax rules.

In [ ]:
financial_rows = []
for i, truth in enumerate(GROUND_TRUTH):
    expected = truth["Quantity"] * truth["Unit_Price"] * (1 + truth["GST_Rate"])
    for method, df in [("Traditional Python", traditional_df), ("AI Extraction", ai_df)]:
        actual = parse_number(df.iloc[i]["Total_Amount"])
        financial_rows.append({
            "Method":method, "Invoice_ID":truth["Invoice_ID"],
            "Expected_Total":round(expected, 2), "Extracted_Total":actual,
            "Amount_Match": actual is not None and abs(expected - actual) < 0.01
        })
financial_validation = pd.DataFrame(financial_rows)
display(financial_validation)
print("Financial validations passed:", int(financial_validation["Amount_Match"].sum()), "/", len(financial_validation))

## 8. Performance benchmark
Warm up the model, then run each method multiple times. Timings are environment-specific and should not be generalized beyond this experiment.

In [ ]:
BENCHMARK_RUNS = 3
_ = extract_invoice_with_ai(research_doc[0])
traditional_times, ai_times = [], []
for _ in range(BENCHMARK_RUNS):
    start = time.perf_counter()
    _ = [extract_traditional_invoice(page) for page in research_doc]
    traditional_times.append(time.perf_counter() - start)
for _ in range(BENCHMARK_RUNS):
    start = time.perf_counter()
    _ = [extract_invoice_with_ai(page) for page in research_doc]
    ai_times.append(time.perf_counter() - start)
benchmark_df = pd.DataFrame({
    "Method":["Traditional Python","AI Extraction"],
    "Average Time (seconds)":[sum(traditional_times)/len(traditional_times), sum(ai_times)/len(ai_times)],
    "Runs":[BENCHMARK_RUNS, BENCHMARK_RUNS]
})
display(benchmark_df)
plt.figure(figsize=(8,5))
plt.bar(benchmark_df["Method"], benchmark_df["Average Time (seconds)"])
plt.title("Average invoice extraction time")
plt.ylabel("Seconds (lower is faster)")
plt.tight_layout()
plt.show()

## 9. Export reproducible outputs
The notebook creates the same named outputs each run. Review the generated CSVs before committing updated snapshots to GitHub.

In [ ]:
os.makedirs("data", exist_ok=True)
os.makedirs("results", exist_ok=True)
traditional_df.to_csv("data/Traditional_Invoice_Extraction.csv", index=False)
ai_df.to_csv("data/AI_Invoice_Extraction.csv", index=False)
truth_df.to_csv("data/Ground_Truth_Invoices.csv", index=False)
field_comparison.to_csv("results/Field_Level_Comparison.csv", index=False)
financial_validation.to_csv("results/Financial_Validation.csv", index=False)
benchmark_df.to_csv("results/Performance_Benchmark.csv", index=False)
summary.to_csv("results/Extraction_Accuracy_Summary.csv", index=False)
print("Exported updated datasets and results.")

## 10. Hybrid AI + Python Gradio application
The application prefers values found by deterministic PDF parsing and uses Qwen as a fallback for fields the rules cannot identify. It reports field sources, missing values, and duplicate invoice IDs.

In [ ]:
def extract_invoice_hybrid(page):
    """
    AI + Python hybrid extraction.
    The deterministic PDF parser takes precedence when it finds a value;
    AI fills any fields the parser cannot read. If AI fails, the parser's
    extracted values are still returned rather than losing the whole page.
    """
    python_record = extract_traditional_invoice(page)
    ai_error = None

    try:
        ai_record = extract_invoice_with_ai(page)
    except Exception as exc:
        ai_record = {field: None for field in FIELDS}
        ai_error = str(exc)

    combined = {}
    field_sources = {}
    for field in FIELDS:
        python_value = python_record.get(field)
        ai_value = ai_record.get(field)

        if python_value is not None and not pd.isna(python_value):
            combined[field] = python_value
            field_sources[field] = "Python rule-based"
        elif ai_value is not None and not pd.isna(ai_value):
            combined[field] = ai_value
            field_sources[field] = "Qwen AI fallback"
        else:
            combined[field] = None
            field_sources[field] = "Missing"

    return combined, field_sources, ai_error


def process_uploaded_invoice(pdf_path):
    if not pdf_path:
        raise gr.Error("Please upload a PDF invoice.")

    try:
        pdf = fitz.open(pdf_path)
    except Exception as exc:
        raise gr.Error(f"Could not open PDF: {exc}")

    records = []
    source_records = []
    ai_errors = []

    try:
        if len(pdf) == 0:
            raise gr.Error("The PDF contains no pages.")

        for page_number, page in enumerate(pdf, start=1):
            try:
                record, field_sources, ai_error = extract_invoice_hybrid(page)
                record["Page"] = page_number
                record["Field_Sources"] = json.dumps(field_sources)
                record["AI_Status"] = "AI extraction completed" if ai_error is None else "Python fallback used; AI error"
                records.append(record)
                source_records.append(field_sources)
                if ai_error:
                    ai_errors.append(f"Page {page_number}: {ai_error[:180]}")
            except Exception as exc:
                raise gr.Error(f"Processing failed on page {page_number}: {exc}")
    finally:
        pdf.close()

    result_df = pd.DataFrame(records)
    result_df["Missing_Fields"] = result_df[FIELDS].isna().sum(axis=1)
    result_df["Duplicate_Invoice_ID"] = (
        result_df["Invoice_ID"].notna()
        & result_df["Invoice_ID"].duplicated(keep=False)
    )

    missing_total = int(result_df[FIELDS].isna().sum().sum())
    duplicate_ids = int(result_df["Invoice_ID"].dropna().duplicated().sum())
    python_fields = sum(
        1 for sources in source_records for source in sources.values()
        if source == "Python rule-based"
    )
    ai_fields = sum(
        1 for sources in source_records for source in sources.values()
        if source == "Qwen AI fallback"
    )

    csv_path = os.path.join(
        tempfile.gettempdir(),
        f"invoice_extraction_{time.time_ns()}.csv"
    )
    result_df.to_csv(csv_path, index=False)

    status = (
        f"Processing completed\\n"
        f"Pages processed: {len(result_df)}\\n"
        f"Missing field values: {missing_total}\\n"
        f"Duplicate invoice IDs: {duplicate_ids}\\n"
        f"Fields supplied by Python rules: {python_fields}\\n"
        f"Fields supplied by Qwen fallback: {ai_fields}\\n"
        "Python values are preferred when explicitly found in the PDF; Qwen fills gaps.\\n"
        "Scanned PDFs require OCR. Review extracted data before business use."
    )
    if ai_errors:
        status += "\\nAI warnings: " + " | ".join(ai_errors[:3])

    return result_df, status, csv_path


with gr.Blocks(title="AI PDF to CSV Automation", theme=gr.themes.Soft()) as demo:
    gr.Markdown(
        "# AI PDF to CSV Automation\\n"
        "Upload a text-based invoice PDF. Python extracts clearly labelled fields, "
        "Qwen fills fields the rules cannot identify, and the app generates a CSV."
    )

    pdf_input = gr.File(
        label="Upload invoice PDF",
        file_types=[".pdf"],
        type="filepath"
    )
    extract_button = gr.Button("Extract Invoice Data", variant="primary")
    extracted_table = gr.Dataframe(
        label="Hybrid Extracted Invoice Data",
        interactive=False,
        wrap=True
    )
    status_output = gr.Textbox(
        label="Processing and Validation Summary",
        lines=9,
        interactive=False
    )
    csv_download = gr.File(label="Download CSV")

    extract_button.click(
        fn=process_uploaded_invoice,
        inputs=pdf_input,
        outputs=[extracted_table, status_output, csv_download]
    )

print("Hybrid Gradio interface created successfully.")

## 11. Launch the application
Run this cell when you want to use the temporary Gradio share link. The link is not permanent and expires; stop the cell to shut down the app.

In [ ]:
demo.launch(share=True, debug=False)